In [1]:
import pandas as pd
df = pd.read_csv("sorted_files_beat_this_4.csv")
df["label"].unique()

array(['2 (Classical)', '3 (Jazz)', '0 + 3', '2 + 3',
       '1 (Pop + HJDB + Hip-Hop)', '0 (Pop + Rock + Metal)', '1 + 3',
       '0 + 1'], dtype=object)

In [ ]:
# creating small cluster for tests
import numpy as np
import os
import shutil
from tqdm import tqdm

cluster_number = 2
SAVE_NPZ = True
trying_set = True 

cluster_name = "try" if trying_set else cluster_number
root = "/hpcwork/ui556004/data/beat_this/"
root_save = "/hpcwork/ui556004/data/beat_this/clusters"
save_spectrograms_path = os.path.join(root_save, f"cluster_{cluster_name}/data/audio/spectrograms")
annotations_path = os.path.join(root_save, f"cluster_{cluster_name}/data/annotations")
os.makedirs(save_spectrograms_path, exist_ok = True)
shutil.copytree("/hpcwork/ui556004/data/beat_this/clusters/annotations", annotations_path, dirs_exist_ok = True)
os.makedirs(os.path.join(root_save, f"cluster_{cluster_number}/data/annotations"), exist_ok = True)
df_filtered = df[df["label"].str.contains(str(cluster_number))]
files = list(df_filtered["file"].values)
# getting all files and corresponding datasets
files_wo_dataset = [file.split("___")[1] for file in files]
datasets_used = set([file.split("___")[0] for file in files])
print(len(df_filtered))
print(datasets_used)
num_files = 5

gtzan_files = 0
for dataset in tqdm(datasets_used):
    if dataset != "gtzan":
        dataset_files = {}
        data_npz = np.load(os.path.join(root, f"{dataset}.npz"))
        lst = data_npz.files
        npz_files_filtered = [file.split("/")[0] for file in lst]
        selected_files = [file for file in npz_files_filtered if file in files_wo_dataset][:5]
        print(f"number of selected files from the {dataset} is {len(selected_files)}")   
        for piece in selected_files:
            # getting all augmentations of the same file
            all_similar = [file for file in lst if piece in file ]
            if (len(all_similar) != 22 and len(all_similar) == 1):
                gtzan_files += 1
                #print(len(all_similar))
            
            pieces = {f"{file}" : data_npz[file] for file in all_similar}
            dataset_files = {**dataset_files, **pieces }
            #cluster = {**cluster, **pieces }
        
    path_to_save = os.path.join(save_spectrograms_path , f"{dataset}.npz")
    if SAVE_NPZ:
        np.savez(path_to_save, **dataset_files)
        print(f"saved npz of {dataset}")
        #datasets_npz[dataset] =dataset_files
    # sanity check: except for gtzan, each file should be repeated 22 times
    #assert (len(cluster) - gtzan_files) / 22 + gtzan_files == len(df_filtered)

1098
{'gtzan', 'hainsworth', 'smc', 'harmonix', 'simac', 'rwc', 'beatles', 'ballroom', 'jaah', 'filosax', 'tapcorrect', 'guitarset', 'asap'}


 15%|█▌        | 2/13 [00:00<00:00, 13.12it/s]

saved npz of gtzan
number of selected files from the hainsworth is 0
saved npz of hainsworth
number of selected files from the smc is 5
saved npz of smc


 38%|███▊      | 5/13 [00:00<00:01,  7.29it/s]

number of selected files from the harmonix is 0
saved npz of harmonix
number of selected files from the simac is 0
saved npz of simac
number of selected files from the rwc is 5


...

number of selected files from the ballroom is 0
saved npz of ballroom
number of selected files from the jaah is 0
saved npz of jaah
number of selected files from the filosax is 5


 92%|█████████▏| 12/13 [00:02<00:00,  4.19it/s]

saved npz of filosax
number of selected files from the tapcorrect is 0
saved npz of tapcorrect
number of selected files from the guitarset is 0
saved npz of guitarset
number of selected files from the asap is 5


100%|██████████| 13/13 [00:03<00:00,  3.96it/s]

saved npz of asap


In [2]:
import numpy as np
import os
from tqdm import tqdm
def prepare_cluster_data(cluster_number, SAVE_NPZ):
    root = "/hpcwork/ui556004/data/beat_this/"
    root_save = "/hpcwork/ui556004/data/beat_this/clusters"
    save_spectrograms_path = os.path.join(root_save, f"cluster_{cluster_number}/data/audio/spectrograms")
    os.makedirs(save_spectrograms_path, exist_ok = True)
    os.makedirs(os.path.join(root_save, f"cluster_{cluster_number}/data/annotations"), exist_ok = True)
    df_filtered = df[df["label"].str.contains(str(cluster_number))]
    files = list(df_filtered["file"].values)
    # getting all files and corresponding datasets
    files_wo_dataset = [file.split("___")[1] for file in files]
    datasets_used = set([file.split("___")[0] for file in files])
    print(len(df_filtered))
    print(datasets_used)

    gtzan_files = 0
    for dataset in tqdm(datasets_used):
        dataset_files = {}
        data_npz = np.load(os.path.join(root, f"{dataset}.npz"))
        lst = data_npz.files
        npz_files_filtered = set([file.split("/")[0] for file in lst])
        selected_files = [file for file in npz_files_filtered if file in files_wo_dataset]
        print(f"number of selected files from the {dataset} is {len(selected_files)}")   
        for piece in selected_files:
            # getting all augmentations of the same file
            all_similar = [file for file in lst if piece in file ]
            if (len(all_similar) != 22 and len(all_similar) == 1):
                gtzan_files += 1
                #print(len(all_similar))
            
            pieces = {f"{file}" : data_npz[file] for file in all_similar}
            dataset_files = {**dataset_files, **pieces }
            #cluster = {**cluster, **pieces }
        
        path_to_save = os.path.join(save_spectrograms_path , f"{dataset}.npz")
        if SAVE_NPZ:
            np.savez(path_to_save, **dataset_files)
            print(f"saved npz of {dataset}")
        #datasets_npz[dataset] =dataset_files
    # sanity check: except for gtzan, each file should be repeated 22 times
    #assert (len(cluster) - gtzan_files) / 22 + gtzan_files == len(df_filtered)

In [ ]:
prepare_cluster_data(cluster_number=1, SAVE_NPZ=True)

1098
{'filosax', 'asap', 'hainsworth', 'smc', 'beatles', 'guitarset', 'simac', 'gtzan', 'rwc', 'harmonix', 'jaah', 'ballroom', 'tapcorrect'}


  0%|          | 0/13 [00:00<?, ?it/s]

number of selected files from the filosax is 8


  8%|▊         | 1/13 [00:02<00:35,  2.94s/it]

saved npz of filosax
number of selected files from the asap is 473


...

saved npz of jaah
number of selected files from the ballroom is 124


 92%|█████████▏| 12/13 [04:04<00:10, 10.43s/it]

saved npz of ballroom
number of selected files from the tapcorrect is 5


100%|██████████| 13/13 [04:05<00:00, 18.89s/it]

saved npz of tapcorrect


In [3]:
prepare_cluster_data(cluster_number=1, SAVE_NPZ=True)

1433
{'candombe', 'hainsworth', 'hjdb', 'ballroom', 'smc', 'gtzan', 'harmonix', 'rwc', 'simac', 'tapcorrect', 'beatles', 'jaah'}


  0%|          | 0/12 [00:00<?, ?it/s]

number of selected files from the candombe is 35


  8%|▊         | 1/12 [00:08<01:33,  8.46s/it]

saved npz of candombe
number of selected files from the hainsworth is 41


...

saved npz of tapcorrect
number of selected files from the beatles is 5


 92%|█████████▏| 11/12 [03:27<00:14, 14.84s/it]

saved npz of beatles
number of selected files from the jaah is 4


100%|██████████| 12/12 [03:28<00:00, 17.39s/it]

saved npz of jaah


In [6]:
import numpy as np
import os
from tqdm import tqdm
import pandas as pd
import shutil
def get_split_files(cluster_number):
    root = "/hpcwork/ui556004/data/beat_this/"
    root_save = "/hpcwork/ui556004/data/beat_this/clusters"
    save_spectrograms_path = os.path.join(root_save, f"cluster_{cluster_number}/data/audio/spectrograms")
    annotations_path = os.path.join(root_save, f"cluster_{cluster_number}/data/annotations")
    os.makedirs(save_spectrograms_path, exist_ok = True)
    #shutil.copytree("/hpcwork/ui556004/data/beat_this/clusters/annotations", annotations_path, dirs_exist_ok = True)
    #os.makedirs(os.path.join(root_save, f"cluster_{cluster_number}/data/annotations"), exist_ok = True)
    # filtering files that belong to cluster number 0
    cluster = {}
    df_filtered = df[df["label"].str.contains(str(cluster_number))]
    files = list(df_filtered["file"].values)
    # getting all files and corresponding datasets
    files_wo_dataset = [file.split("___")[1] for file in files]
    datasets_used = set([file.split("___")[0] for file in files])
    train_val_split = {}
    for dataset in tqdm(datasets_used):
        validation_files_new = 0
        train_files_new = 0
        data_npz = np.load(os.path.join(root, f"{dataset}.npz"))
        lst = data_npz.files
        npz_files_filtered = set([file.split("/")[0] for file in lst])
        selected_files = [file for file in npz_files_filtered if file in files_wo_dataset]
        #print(f"number of selected files from the {dataset} is {len(selected_files)}")   
        if dataset != "gtzan":
            split = pd.read_csv(f"/hpcwork/ui556004/data/beat_this/clusters/cluster_0/data/annotations/{dataset}/single.split", sep = "\t", header = None, names = ["File", "Split"])
            split_filtered = split[split["Split"] != "train"]
            val_files_split = list(split_filtered["File"].values)
            for piece in selected_files:
                if piece in val_files_split:
                    validation_files_new += 1
                else:
                    train_files_new += 1
            train_val_split[dataset] =  (validation_files_new, train_files_new)
        
    return train_val_split
    


In [17]:

root_save = "/hpcwork/ui556004/data/beat_this/clusters"
shutil.copytree("/hpcwork/ui556004/data/beat_this/clusters/annotations", os.path.join(root_save, f"cluster_{0}/data/annotations"), dirs_exist_ok = True)

'/hpcwork/ui556004/data/beat_this/clusters/cluster_0/data/annotations'

In [8]:
get_split_files(2)

100%|██████████| 13/13 [00:00<00:00, 19.26it/s]


{'tapcorrect': (2, 3),
 'ballroom': (16, 108),
 'rwc': (16, 67),
 'asap': (65, 408),
 'hainsworth': (10, 52),
 'simac': (0, 80),
 'guitarset': (0, 4),
 'filosax': (1, 7),
 'jaah': (1, 2),
 'harmonix': (0, 1),
 'smc': (0, 143),
 'beatles': (0, 1)}

In [2]:
def get_split_percentage(train_val_split):
    validation_items = 0 #4
    train_items = 0
    for keys, values in train_val_split.items():
        validation_items += values[0]
        train_items += values[1]
    print(validation_items)
    print(train_items)
    print(validation_items / train_items * 100)

In [20]:
for cluster_number in range(4):
    train_val_split = get_split_files(cluster_number)
    print(f"results for the cluster {cluster_number}")
    get_split_percentage(train_val_split)

100%|██████████| 12/12 [00:01<00:00, 11.32it/s]


results for the cluster 0
119
794
14.987405541561714


100%|██████████| 12/12 [00:00<00:00, 22.94it/s]


results for the cluster 1
179
958
18.684759916492695


100%|██████████| 13/13 [00:00<00:00, 20.76it/s]


results for the cluster 2
111
876
12.67123287671233


100%|██████████| 13/13 [00:00<00:00, 26.08it/s]

results for the cluster 3
112
1298
8.628659476117104


In [30]:
validation_items = 0 #4
train_items = 0
for keys, values in train_val_split.items():
    validation_items += values[0]
    train_items += values[1]
print(validation_items)
print(train_items)
print(validation_items / train_items * 100)


112
731
15.321477428180575


In [28]:
validation_items = 0  #3
train_items = 0
for keys, values in train_val_split.items():
    validation_items += values[0]
    train_items += values[1]
print(validation_items)
print(train_items)
print(validation_items / train_items * 100)


147
781
18.82202304737516


In [ ]:
validation_items = 0  # 2
train_items = 0
for keys, values in train_val_split.items():
    validation_items += values[0]
    train_items += values[1]
print(validation_items)
print(train_items)
print(validation_items / train_items * 100)


101
597
16.917922948073702


In [ ]:
validation_items = 0 # cluster 1
train_items = 0
for keys, values in train_val_split.items():
    validation_items += values[0]
    train_items += values[1]
print(validation_items)
print(train_items)
print(validation_items / train_items * 100)


50
899
5.561735261401557


In [ ]:
validation_items = 0   # 0
train_items = 0
for keys, values in train_val_split.items():
    validation_items += values[0]
    train_items += values[1]
print(validation_items)
print(train_items)
print(validation_items / train_items * 100)


103
818
12.591687041564793


In [5]:
datasets_npz.keys()

dict_keys(['rwc', 'gtzan', 'smc', 'guitarset', 'jaah', 'tapcorrect', 'harmonix', 'beatles', 'ballroom', 'simac', 'asap', 'hainsworth'])

In [24]:
np.savez("spectrograms.npz", spectrograms=spectrograms)

In [ ]:
dataset = "gtzan"
spectrograms = datasets_npz[dataset]
#spectrograms = {str(k): v for k, v in spectrograms.items()}
save_spectrograms_path = os.path.join(root_save, f"cluster_{cluster_number}/data/audio/spectrograms")
path_to_save = os.path.join(save_spectrograms_path , f"{dataset}.npz")
np.savez(path_to_save, **spectrograms)

In [4]:
import numpy as np
data = np.load("/hpcwork/ui556004/data/beat_this/clusters/cluster_1/data/audio/spectrograms/harmonix.npz")
lst = data.files
len(lst) / 22

487.0

In [47]:
dataset, remainder = lst[0].split("/", 1)
remainder

'track'

In [ ]:
import numpy as np
import os
from tqdm import tqdm
root = "/hpcwork/ui556004/data/beat_this/"
# filtering files that belong to cluster number 0
cluster = {}
df_filtered = df[df["label"].str.contains("0")]
files = list(df_filtered["file"].values)
# getting all files and corresponding datasets
files_wo_dataset = [file.split("___")[1] for file in files]
datasets_used = set([file.split("___")[0] for file in files])
print(len(df_filtered))
print(datasets_used)

gtzan_files = 0
cluster = {}
for dataset in tqdm(datasets_used):
#dataset = "asap"
    data_npz = np.load(os.path.join(root, f"{dataset}.npz"))
    lst = data_npz.files
    npz_files_filtered = set([file.split("/")[0] for file in lst])
    selected_files = [file for file in npz_files_filtered if file in files_wo_dataset]
    print(f"number of selected files from the {dataset} is {len(selected_files)}")   
    for piece in selected_files:
        # getting all augmentations of the same file
        all_similar = [file for file in lst if piece in file ]
        if (len(all_similar) != 22 and len(all_similar) == 1):
            gtzan_files += 1
            #print(len(all_similar))
        
        pieces = {f"{dataset}___{file}" : data_npz[file] for file in all_similar}
        cluster = {**cluster, **pieces }
# sanity check: except for gtzan, each file should be repeated 22 times
assert (len(cluster) - gtzan_files) / 22 + gtzan_files == len(df_filtered)

In [19]:
import pandas as pd
split = pd.read_csv("/hpcwork/ui556004/data/beat_this/clusters/cluster_0/data/annotations/smc/single.split", sep = "\t", header = None, names = ["File", "Split"])
# split_filtered = split[split["Split"] != "train"]
# list(split_filtered["File"].values)
split

,File,Split
0,smc_001,train
1,smc_002,train
2,smc_003,train
3,smc_004,train
4,smc_005,train
...,...,...
212,smc_285,train
213,smc_286,train
214,smc_287,train
215,smc_288,train


In [1]:
import os 
checkpoint_path = "/hpcwork/ui556004/results/beat_this/checkpoints"
checkpoint_folder = os.path.join(checkpoint_path, "lalala")
os.makedirs(checkpoint_folder, exist_ok = True)